In [4]:
# ============================================================
# NOTEBOOK 05 — NUDGE DECISION ENGINE
# ============================================================

import pandas as pd
from pathlib import Path

project_root = Path.cwd().parent

data_path = (
    project_root
    / "data"
    / "processed"
    / "rule_engine_output.csv"
)

behavior = pd.read_csv(data_path)

print("Dataset shape:", behavior.shape)
print("Personas:", behavior["Persona"].nunique())

print("\nRequired columns:")

required_cols = [
    "Financial_Index",
    "Engagement_Index",
    "Investor_Profile_Index",
    "Financial_Level",
    "Engagement_Level",
    "Profile_Level",
    "Persona",
    "Tier",
    "Priority",
    "Action",
    "Rule_Match_Status"
]

for col in required_cols:
    print(f"{col}: {'✓' if col in behavior.columns else '✗'}")

Dataset shape: (11162, 45)
Personas: 9

Required columns:
Financial_Index: ✓
Engagement_Index: ✓
Investor_Profile_Index: ✓
Financial_Level: ✓
Engagement_Level: ✓
Profile_Level: ✓
Persona: ✓
Tier: ✓
Priority: ✓
Action: ✓
Rule_Match_Status: ✓


In [5]:
# ============================================================
# EXISTING RULE-ENGINE DECISIONS
# ============================================================

decision_map = (
    behavior[
        [
            "Persona",
            "Tier",
            "Priority",
            "Action"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["Tier", "Priority", "Persona"]
    )
)

print("Unique persona decisions:", len(decision_map))
print()

print(
    decision_map.to_string(index=False)
)

Unique persona decisions: 15

              Persona   Tier Priority                                                       Action
      Growth Investor Tier 1     High                       Upsell high-growth investment products
      Growth Investor Tier 1     High Upsell high-growth investment products; increase touchpoints
      Growth Investor Tier 1     High          Upsell growth products; review profile data quality
     Premium Investor Tier 1     High        Assign dedicated RM; offer premium/exclusive products
    Balanced Investor Tier 2   Medium                       Cross-sell balanced portfolio products
     General Investor Tier 2   Medium                     Nurture with periodic offers and reviews
     General Investor Tier 2   Medium          Nurture with periodic offers; monitor profile trend
    Emerging Investor Tier 3   Medium                    Educational content and onboarding nudges
   Potential Investor Tier 3   Medium                                 Targeted 

In [6]:
# ============================================================
# ACTION VARIATION BY PERSONA
# ============================================================

action_summary = (
    behavior[
        [
            "Financial_Level",
            "Engagement_Level",
            "Profile_Level",
            "Persona",
            "Tier",
            "Priority",
            "Action"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "Persona",
            "Financial_Level",
            "Engagement_Level",
            "Profile_Level"
        ]
    )
)

print(action_summary.to_string(index=False))

Financial_Level Engagement_Level Profile_Level               Persona   Tier Priority                                                       Action
           High           Medium        Medium     Balanced Investor Tier 2   Medium                       Cross-sell balanced portfolio products
         Medium             High          High     Balanced Investor Tier 2   Medium                       Cross-sell balanced portfolio products
         Medium             High        Medium     Balanced Investor Tier 2   Medium                       Cross-sell balanced portfolio products
         Medium           Medium          High     Balanced Investor Tier 2   Medium                       Cross-sell balanced portfolio products
           High              Low          High Dormant Wealth Holder Tier 4     High       Reactivation campaign - high value; senior RM outreach
           High              Low           Low Dormant Wealth Holder Tier 4     High       Reactivation campaign - high valu

In [7]:
# ============================================================
# RULE-CELL ACTION VALIDATION
# ============================================================

rule_cells = (
    behavior[
        [
            "Financial_Level",
            "Engagement_Level",
            "Profile_Level",
            "Persona",
            "Action"
        ]
    ]
    .drop_duplicates()
)

cell_counts = (
    rule_cells
    .groupby(
        [
            "Financial_Level",
            "Engagement_Level",
            "Profile_Level"
        ]
    )
    .size()
    .reset_index(name="Action_Count")
)

print("Total unique rule cells:", len(cell_counts))

print("\nCells with more than one action:")
print(
    cell_counts[cell_counts["Action_Count"] > 1]
    .to_string(index=False)
)

print("\nCells with exactly one action:",
      (cell_counts["Action_Count"] == 1).sum())

print("Cells with multiple actions:",
      (cell_counts["Action_Count"] > 1).sum())

Total unique rule cells: 27

Cells with more than one action:
Empty DataFrame
Columns: [Financial_Level, Engagement_Level, Profile_Level, Action_Count]
Index: []

Cells with exactly one action: 27
Cells with multiple actions: 0


In [8]:
# ============================================================
# RULE-CELL ACTION VALIDATION
# ============================================================

rule_cells = (
    behavior[
        [
            "Financial_Level",
            "Engagement_Level",
            "Profile_Level",
            "Persona",
            "Action"
        ]
    ]
    .drop_duplicates()
)

cell_counts = (
    rule_cells
    .groupby(
        [
            "Financial_Level",
            "Engagement_Level",
            "Profile_Level"
        ]
    )
    .size()
    .reset_index(name="Action_Count")
)

print("Total unique rule cells:", len(cell_counts))

print("\nCells with more than one action:")
print(
    cell_counts[cell_counts["Action_Count"] > 1]
    .to_string(index=False)
)

print(
    "\nCells with exactly one action:",
    (cell_counts["Action_Count"] == 1).sum()
)

print(
    "Cells with multiple actions:",
    (cell_counts["Action_Count"] > 1).sum()
)

Total unique rule cells: 27

Cells with more than one action:
Empty DataFrame
Columns: [Financial_Level, Engagement_Level, Profile_Level, Action_Count]
Index: []

Cells with exactly one action: 27
Cells with multiple actions: 0


In [9]:
# ============================================================
# NUDGE OBJECTIVE MAPPING
# ============================================================

nudge_objectives = {

    "Upsell high-growth investment products":
        "Increase investment depth",

    "Upsell growth products; review profile data quality":
        "Increase investment depth and improve profile accuracy",

    "Upsell high-growth investment products; increase touchpoints":
        "Increase investment depth and strengthen engagement",

    "Assign dedicated RM; offer premium/exclusive products":
        "Maximize high-value customer relationship",

    "Cross-sell balanced portfolio products":
        "Increase portfolio diversification",

    "Nurture with periodic offers and reviews":
        "Maintain engagement and encourage repeat investment",

    "Nurture with periodic offers; monitor profile trend":
        "Maintain engagement and monitor behavioral change",

    "Educational content and onboarding nudges":
        "Build investment confidence and encourage conversion",

    "Targeted conversion campaign":
        "Convert investment interest into active investment",

    "Targeted conversion campaign; re-engage on low activity":
        "Convert interest while addressing declining activity",

    "Reactivation campaign - high value":
        "Reactivate high-value inactive customers",

    "Reactivation campaign - high value; senior RM outreach":
        "Reactivate high-value customers through personalized intervention",

    "Low-cost reactivation or deprioritize":
        "Re-engage efficiently while controlling acquisition cost",

    "Reactivation campaign - medium value":
        "Reactivate medium-value inactive customers",

    "Reactivation campaign - medium value; priority follow-up":
        "Prioritize reactivation of valuable inactive customers"
}

behavior["Nudge_Objective"] = behavior["Action"].map(nudge_objectives)

print(
    "Missing nudge objectives:",
    behavior["Nudge_Objective"].isna().sum()
)

Missing nudge objectives: 0


In [10]:
# ============================================================
# RECOMMENDED NUDGE / INTERVENTION
# ============================================================

nudge_interventions = {

    "Increase investment depth":
        "Recommend suitable higher-value or growth-oriented investment products",

    "Increase investment depth and improve profile accuracy":
        "Review customer profile data and provide a suitable growth-product recommendation",

    "Increase investment depth and strengthen engagement":
        "Provide a growth-product recommendation with additional relationship touchpoints",

    "Maximize high-value customer relationship":
        "Dedicated relationship-manager outreach with premium/exclusive product recommendations",

    "Increase portfolio diversification":
        "Recommend complementary gold/silver investment options to diversify holdings",

    "Maintain engagement and encourage repeat investment":
        "Periodic portfolio review and personalized investment offers",

    "Maintain engagement and monitor behavioral change":
        "Periodic review with behavioral trend monitoring and targeted follow-up",

    "Build investment confidence and encourage conversion":
        "Provide educational investment content followed by an onboarding nudge",

    "Convert investment interest into active investment":
        "Targeted conversion message with a relevant low-friction investment opportunity",

    "Convert interest while addressing declining activity":
        "Re-engagement message followed by a targeted investment conversion offer",

    "Reactivate high-value inactive customers":
        "Personalized reactivation campaign focused on returning high-value customers",

    "Reactivate high-value customers through personalized intervention":
        "Senior relationship-manager outreach with a personalized reactivation conversation",

    "Re-engage efficiently while controlling acquisition cost":
        "Low-cost digital reactivation message or deprioritize based on expected value",

    "Reactivate medium-value inactive customers":
        "Standard reactivation campaign with a relevant investment reminder",

    "Prioritize reactivation of valuable inactive customers":
        "Priority follow-up with personalized reactivation messaging"
}

behavior["Recommended_Intervention"] = (
    behavior["Nudge_Objective"].map(nudge_interventions)
)

print(
    "Missing interventions:",
    behavior["Recommended_Intervention"].isna().sum()
)

Missing interventions: 0


In [11]:
# ============================================================
# TARGET KPI
# ============================================================

kpi_mapping = {

    "Increase investment depth":
        "Average Investment per User",

    "Increase investment depth and improve profile accuracy":
        "Average Investment per User",

    "Increase investment depth and strengthen engagement":
        "Average Investment per User",

    "Maximize high-value customer relationship":
        "Average Wallet Holding",

    "Increase portfolio diversification":
        "Gold vs Silver Preference Ratio",

    "Maintain engagement and encourage repeat investment":
        "Repeat Purchase Rate",

    "Maintain engagement and monitor behavioral change":
        "Retention Rate",

    "Build investment confidence and encourage conversion":
        "KYC Completion Rate",

    "Convert investment interest into active investment":
        "Current Month Active Investors",

    "Convert interest while addressing declining activity":
        "Current Month Active Investors",

    "Reactivate high-value inactive customers":
        "Dormant User Reactivation Rate",

    "Reactivate high-value customers through personalized intervention":
        "Dormant User Reactivation Rate",

    "Re-engage efficiently while controlling acquisition cost":
        "Dormant User Reactivation Rate",

    "Reactivate medium-value inactive customers":
        "Dormant User Reactivation Rate",

    "Prioritize reactivation of valuable inactive customers":
        "Dormant User Reactivation Rate"
}

behavior["Target_KPI"] = behavior["Nudge_Objective"].map(kpi_mapping)

print(
    "Missing KPIs:",
    behavior["Target_KPI"].isna().sum()
)

Missing KPIs: 0


In [12]:
# ============================================================
# 27-CELL NUDGE DECISION MATRIX
# ============================================================

decision_matrix = (
    behavior[
        [
            "Financial_Level",
            "Engagement_Level",
            "Profile_Level",
            "Persona",
            "Tier",
            "Priority",
            "Action",
            "Nudge_Objective",
            "Recommended_Intervention",
            "Target_KPI"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "Tier",
            "Financial_Level",
            "Engagement_Level",
            "Profile_Level"
        ]
    )
    .reset_index(drop=True)
)

print("Decision matrix rows:", len(decision_matrix))

print(
    decision_matrix.to_string(index=False)
)

Decision matrix rows: 27
Financial_Level Engagement_Level Profile_Level               Persona   Tier Priority                                                       Action                                                   Nudge_Objective                                                               Recommended_Intervention                      Target_KPI
           High             High          High      Premium Investor Tier 1     High        Assign dedicated RM; offer premium/exclusive products                         Maximize high-value customer relationship Dedicated relationship-manager outreach with premium/exclusive product recommendations          Average Wallet Holding
           High             High           Low       Growth Investor Tier 1     High          Upsell growth products; review profile data quality            Increase investment depth and improve profile accuracy      Review customer profile data and provide a suitable growth-product recommendation     Average In

In [13]:
# ============================================================
# DECISION MATRIX VALIDATION
# ============================================================

print("Total decision cells:", len(decision_matrix))

print(
    "Missing objectives:",
    decision_matrix["Nudge_Objective"].isna().sum()
)

print(
    "Missing interventions:",
    decision_matrix["Recommended_Intervention"].isna().sum()
)

print(
    "Missing KPIs:",
    decision_matrix["Target_KPI"].isna().sum()
)

print(
    "Unique personas:",
    decision_matrix["Persona"].nunique()
)

print(
    "Unique actions:",
    decision_matrix["Action"].nunique()
)

Total decision cells: 27
Missing objectives: 0
Missing interventions: 0
Missing KPIs: 0
Unique personas: 9
Unique actions: 15


In [14]:
# ============================================================
# SAVE DECISION MATRIX
# ============================================================

decision_matrix_path = (
    project_root
    / "data"
    / "processed"
    / "nudge_decision_matrix.csv"
)

decision_matrix.to_csv(
    decision_matrix_path,
    index=False
)

print("Saved:")
print(decision_matrix_path)

Saved:
d:\NudgeIQ\data\processed\nudge_decision_matrix.csv


In [15]:
# ============================================================
# FINAL CUSTOMER-LEVEL NUDGE OUTPUT
# ============================================================

final_columns = [
    "age",
    "job",
    "marital",
    "education",
    "balance",
    "Persona",
    "Tier",
    "Priority",
    "Action",
    "Nudge_Objective",
    "Recommended_Intervention",
    "Target_KPI"
]

customer_nudge_output = behavior[final_columns].copy()

customer_nudge_path = (
    project_root
    / "data"
    / "processed"
    / "customer_nudge_recommendations.csv"
)

customer_nudge_output.to_csv(
    customer_nudge_path,
    index=False
)

print("Saved:")
print(customer_nudge_path)

print("\nShape:", customer_nudge_output.shape)

Saved:
d:\NudgeIQ\data\processed\customer_nudge_recommendations.csv

Shape: (11162, 12)


In [16]:
# ============================================================
# CUSTOMER-LEVEL NUDGE OUTPUT VALIDATION
# ============================================================

print("=== CUSTOMER NUDGE OUTPUT VALIDATION ===")

print("Customers:", len(customer_nudge_output))

print(
    "Missing Persona:",
    customer_nudge_output["Persona"].isna().sum()
)

print(
    "Missing Action:",
    customer_nudge_output["Action"].isna().sum()
)

print(
    "Missing Nudge Objective:",
    customer_nudge_output["Nudge_Objective"].isna().sum()
)

print(
    "Missing Intervention:",
    customer_nudge_output["Recommended_Intervention"].isna().sum()
)

print(
    "Missing Target KPI:",
    customer_nudge_output["Target_KPI"].isna().sum()
)

print(
    "\nUnique personas:",
    customer_nudge_output["Persona"].nunique()
)

print(
    "Unique actions:",
    customer_nudge_output["Action"].nunique()
)

=== CUSTOMER NUDGE OUTPUT VALIDATION ===
Customers: 11162
Missing Persona: 0
Missing Action: 0
Missing Nudge Objective: 0
Missing Intervention: 0
Missing Target KPI: 0

Unique personas: 9
Unique actions: 15


In [17]:
# ============================================================
# NUDGE PRIORITIZATION
# ============================================================

tier_weight = {
    "Tier 1": 1.00,
    "Tier 2": 0.75,
    "Tier 3": 0.50,
    "Tier 4": 0.25
}

priority_weight = {
    "High": 1.00,
    "Medium": 0.60,
    "Low": 0.30
}

behavior["Tier_Score"] = behavior["Tier"].map(tier_weight)
behavior["Priority_Score"] = behavior["Priority"].map(priority_weight)

# Normalize the three behavioral indices to 0–1
for col in [
    "Financial_Index",
    "Engagement_Index",
    "Investor_Profile_Index"
]:
    min_val = behavior[col].min()
    max_val = behavior[col].max()

    behavior[f"{col}_Normalized"] = (
        (behavior[col] - min_val)
        / (max_val - min_val)
    )

# ------------------------------------------------------------
# NUDGE PRIORITY SCORE
# ------------------------------------------------------------

behavior["Nudge_Priority_Score"] = (
    0.30 * behavior["Tier_Score"]
    + 0.20 * behavior["Priority_Score"]
    + 0.20 * behavior["Financial_Index_Normalized"]
    + 0.15 * behavior["Engagement_Index_Normalized"]
    + 0.15 * behavior["Investor_Profile_Index_Normalized"]
)

print(
    behavior["Nudge_Priority_Score"]
    .describe()
)

count    11162.000000
mean         0.335695
std          0.105698
min          0.156055
25%          0.242345
50%          0.334492
75%          0.407471
max          0.838368
Name: Nudge_Priority_Score, dtype: float64


In [18]:
# ============================================================
# PRIORITY BANDS
# ============================================================

behavior["Nudge_Priority_Band"] = pd.cut(
    behavior["Nudge_Priority_Score"],
    bins=[-float("inf"), 0.35, 0.60, float("inf")],
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

print(
    behavior["Nudge_Priority_Band"]
    .value_counts()
    .sort_index()
)

Nudge_Priority_Band
Low       6249
Medium    4807
High       106
Name: count, dtype: int64


In [19]:
# ============================================================
# TOP NUDGE PRIORITY CUSTOMERS
# ============================================================

top_customers = (
    behavior[
        [
            "Persona",
            "Tier",
            "Priority",
            "Action",
            "Nudge_Objective",
            "Target_KPI",
            "Nudge_Priority_Score",
            "Nudge_Priority_Band"
        ]
    ]
    .sort_values(
        "Nudge_Priority_Score",
        ascending=False
    )
    .head(20)
)

print(
    top_customers.to_string(index=False)
)

         Persona   Tier Priority                                                       Action                                        Nudge_Objective                  Target_KPI  Nudge_Priority_Score Nudge_Priority_Band
Premium Investor Tier 1     High        Assign dedicated RM; offer premium/exclusive products              Maximize high-value customer relationship      Average Wallet Holding              0.838368                High
Premium Investor Tier 1     High        Assign dedicated RM; offer premium/exclusive products              Maximize high-value customer relationship      Average Wallet Holding              0.762700                High
 Growth Investor Tier 1     High Upsell high-growth investment products; increase touchpoints    Increase investment depth and strengthen engagement Average Investment per User              0.760359                High
 Growth Investor Tier 1     High Upsell high-growth investment products; increase touchpoints    Increase investment depth a

In [20]:
# ============================================================
# NUDGE PRIORITY — TIER VALIDATION
# ============================================================

tier_summary = (
    behavior
    .groupby("Tier", observed=True)["Nudge_Priority_Score"]
    .agg(
        Customers="count",
        Mean="mean",
        Median="median",
        Minimum="min",
        Maximum="max"
    )
    .round(3)
)

print(tier_summary)

        Customers   Mean  Median  Minimum  Maximum
Tier                                              
Tier 1        656  0.588   0.583    0.559    0.838
Tier 2       4024  0.410   0.408    0.374    0.541
Tier 3       2094  0.333   0.330    0.311    0.503
Tier 4       4388  0.231   0.237    0.156    0.451


In [21]:
# ============================================================
# NUDGE PRIORITY — PERSONA VALIDATION
# ============================================================

persona_priority = (
    behavior
    .groupby("Persona", observed=True)["Nudge_Priority_Score"]
    .agg(
        Customers="count",
        Mean="mean",
        Median="median",
        Minimum="min",
        Maximum="max"
    )
    .sort_values("Mean", ascending=False)
    .round(3)
)

print(persona_priority)

                       Customers   Mean  Median  Minimum  Maximum
Persona                                                          
Premium Investor             223  0.599   0.593    0.574    0.838
Growth Investor              433  0.582   0.578    0.559    0.760
Balanced Investor           2151  0.419   0.416    0.400    0.541
General Investor            1873  0.399   0.399    0.374    0.415
Potential Investor           926  0.341   0.337    0.322    0.503
Dormant Wealth Holder        555  0.334   0.330    0.311    0.451
Emerging Investor           1168  0.327   0.326    0.311    0.444
Inactive Investor           2378  0.241   0.241    0.218    0.303
Low Engagement User         1455  0.175   0.175    0.156    0.198


In [22]:
# ============================================================
# HIGH NUDGE PRIORITY COMPOSITION
# ============================================================

high_priority = behavior[
    behavior["Nudge_Priority_Band"] == "High"
]

print("High-priority customers:", len(high_priority))

print("\nPersona distribution:")
print(
    high_priority["Persona"]
    .value_counts()
)

print("\nTier distribution:")
print(
    high_priority["Tier"]
    .value_counts()
)

print("\nExisting Priority distribution:")
print(
    high_priority["Priority"]
    .value_counts()
)

High-priority customers: 106

Persona distribution:
Persona
Premium Investor    70
Growth Investor     36
Name: count, dtype: int64

Tier distribution:
Tier
Tier 1    106
Name: count, dtype: int64

Existing Priority distribution:
Priority
High    106
Name: count, dtype: int64


In [23]:
# ============================================================
# WITHIN-TIER NUDGE DIFFERENTIATION
# ============================================================

within_tier = (
    behavior
    .groupby(
        ["Tier", "Priority"],
        observed=True
    )["Nudge_Priority_Score"]
    .agg(
        Customers="count",
        Mean="mean",
        Std="std",
        Minimum="min",
        Maximum="max"
    )
    .round(3)
)

print(within_tier)

                 Customers   Mean    Std  Minimum  Maximum
Tier   Priority                                           
Tier 1 High            656  0.588  0.022    0.559    0.838
Tier 2 Medium         4024  0.410  0.015    0.374    0.541
Tier 3 Medium         2094  0.333  0.016    0.311    0.503
Tier 4 High            555  0.334  0.016    0.311    0.451
       Low            1455  0.175  0.007    0.156    0.198
       Medium         2378  0.241  0.008    0.218    0.303


In [24]:
# ============================================================
# SCORE VARIATION WITHIN EACH TIER
# ============================================================

for tier in ["Tier 1", "Tier 2", "Tier 3", "Tier 4"]:
    
    subset = behavior[
        behavior["Tier"] == tier
    ]["Nudge_Priority_Score"]

    print(
        f"{tier}: "
        f"range={subset.min():.3f} → {subset.max():.3f}, "
        f"std={subset.std():.3f}"
    )

Tier 1: range=0.559 → 0.838, std=0.022
Tier 2: range=0.374 → 0.541, std=0.015
Tier 3: range=0.311 → 0.503, std=0.016
Tier 4: range=0.156 → 0.451, std=0.050


In [26]:
# ============================================================
# FINAL NUDGE OUTPUT WITH PRIORITIZATION
# ============================================================

final_columns = [
    "age",
    "job",
    "marital",
    "education",
    "balance",
    "Persona",
    "Tier",
    "Priority",
    "Action",
    "Nudge_Objective",
    "Recommended_Intervention",
    "Target_KPI",
    "Nudge_Priority_Score",
    "Nudge_Priority_Band"
]

customer_nudge_output = behavior[final_columns].copy()

print("Final customer output shape:", customer_nudge_output.shape)

print(
    "\nMissing values:"
)

print(
    customer_nudge_output[
        [
            "Persona",
            "Action",
            "Nudge_Objective",
            "Recommended_Intervention",
            "Target_KPI",
            "Nudge_Priority_Score",
            "Nudge_Priority_Band"
        ]
    ]
    .isna()
    .sum()
)

Final customer output shape: (11162, 14)

Missing values:
Persona                     0
Action                      0
Nudge_Objective             0
Recommended_Intervention    0
Target_KPI                  0
Nudge_Priority_Score        0
Nudge_Priority_Band         0
dtype: int64


In [27]:
# ============================================================
# SAVE FINAL NUDGEIQ CUSTOMER OUTPUT
# ============================================================

final_output_path = (
    project_root
    / "data"
    / "processed"
    / "final_nudgeiq_output.csv"
)

customer_nudge_output.to_csv(
    final_output_path,
    index=False
)

print("\nSaved final NudgeIQ output:")
print(final_output_path)


Saved final NudgeIQ output:
d:\NudgeIQ\data\processed\final_nudgeiq_output.csv


In [28]:
# ============================================================
# NOTEBOOK 05 — FINAL VALIDATION SUMMARY
# ============================================================

print("=" * 65)
print("NUDGEIQ DECISION ENGINE — FINAL VALIDATION")
print("=" * 65)

print(f"Customers: {len(customer_nudge_output):,}")
print(f"Personas: {customer_nudge_output['Persona'].nunique()}")
print(f"Actions: {customer_nudge_output['Action'].nunique()}")

print("\nDecision fields with missing values:")

validation_cols = [
    "Persona",
    "Action",
    "Nudge_Objective",
    "Recommended_Intervention",
    "Target_KPI",
    "Nudge_Priority_Score",
    "Nudge_Priority_Band"
]

missing = customer_nudge_output[validation_cols].isna().sum()

print(missing)

print("\nPriority distribution:")
print(
    customer_nudge_output["Nudge_Priority_Band"]
    .value_counts()
)

print("\nTier distribution:")
print(
    customer_nudge_output["Tier"]
    .value_counts()
)

print("\n" + "=" * 65)
print("STATUS: NUDGEIQ DECISION ENGINE VALIDATED")
print("=" * 65)

NUDGEIQ DECISION ENGINE — FINAL VALIDATION
Customers: 11,162
Personas: 9
Actions: 15

Decision fields with missing values:
Persona                     0
Action                      0
Nudge_Objective             0
Recommended_Intervention    0
Target_KPI                  0
Nudge_Priority_Score        0
Nudge_Priority_Band         0
dtype: int64

Priority distribution:
Nudge_Priority_Band
Low       6249
Medium    4807
High       106
Name: count, dtype: int64

Tier distribution:
Tier
Tier 4    4388
Tier 2    4024
Tier 3    2094
Tier 1     656
Name: count, dtype: int64

STATUS: NUDGEIQ DECISION ENGINE VALIDATED
